Assignment 3
Compare BERT to interpretable classifiers for text classification

1. Download the ReviewBase dataset
Split dotat on space character
The data set also contains a class labael associated with each text

In [8]:
file_training = open('ReviewBaseTraining.txt', 'r', encoding = 'utf-8')
training_data_string = file_training.read()
file_training.close()

file_validation = open('ReviewBaseValidation.txt', 'r', encoding = 'utf-8')
validation_data_string = file_validation.read()
file_validation.close()

file_test = open('ReviewBaseTest.txt', 'r', encoding = 'utf-8')
test_data_string = file_test.read()
file_test.close()


In [ ]:
training_data = training_data_string.splitlines()
validation_data = validation_data_string.splitlines()
test_data = test_data_string.splitlines()




def split_labels(data):
    current_label = None
    x = []
    y = []
    for word in data:
        label, text = word.split("\t")
        x.append(text)
        y.append(label)
    return x, y

x_train, y_train = split_labels(training_data)
x_val, y_val = split_labels(validation_data)
x_test, y_test = split_labels(test_data)

print(x_train[:2])
print(y_train[:2])



["0\tStory of a man who has unnatural feelings for a pig . Starts out with a opening scene that is a terrific example of absurd comedy . A formal orchestra audience is turned into an insane , violent mob by the crazy chantings of it ' s singers . Unfortunately it stays absurd the WHOLE time with no general narrative eventually making it just too off putting . Even those from the era should be turned off . The cryptic dialogue would make Shakespeare seem easy to a third grader . On a technical level it ' s better than you might think with some good cinematography by future great Vilmos Zsigmond . Future stars Sally Kirkland and Frederic Forrest can be seen briefly .", '1\tBromwell High is a cartoon comedy . It ran at the same time as some other programs about school life , such as " Teachers " . My 35 years in the teaching profession lead me to believe that Bromwell High \' s satire is much closer to reality than is " Teachers " . The scramble to survive financially , the insightful stu

Set up and fine-tune the DeBERTa-v3 model using low-rank adaptiation(LoRA) 
Use a single fully connected layer (after transformer bloacks)


In [ ]:
!pip install -q transformers[sentencepiece] accelerate
pip install peft

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW

from peft import LoraConfig, get_peft_model

from torch.utils.data import Dataset, DataLoader

In [ ]:
#Fine tuning involves setting suitable values for the weights int he feedforward layer
#Slightly modifying weights in BERT itself
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "microsoft/deberta-v3-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

train_data = (x_train, y_train)
dataloaded_train = DataLoader(train_data, batch_size = 32, shuffle = True)

test_data = (x_test, y_test)
validation_data = (x_val, y_val)
data_loader_test = DataLoader(test_data, batch_size = 32, shuffle = False)
data_loader_val = DataLoader(validation_data, batch_size = 32, shuffle = False)

train_x, train_y = tokenizer(dataloaded_train, return_tensors="pt", truncation = True, padding = True, max_length = 512).to(device)
test_x, test_y = tokenizer(data_loader_test, return_tensors="pt", truncation = True, padding = True, max_length = 512).to(device)
val_x, val_y = tokenizer(data_loader_val, return_tensors="pt", truncation = True, padding = True, max_length = 512).to(device)

lora_config = LoraConfig(r = 8, target_modules = ["query_proj", "key_proj", "value_proj"])

model = get_peft_model(model, lora_config)

for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear) and ("attn" in name.lower() or "attention" in name.lower()):
        print(name)

In [ ]:
#Fine tuning involves setting suitable values for the weights int he feedforward layer
#Slightly modifying weights in BERT itself

model_name = "microsoft/deberta-v3-small"

tokenizer = AutoTokenizer.from_pretrained(model_name) #Picks the tokenizer
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2) #Loads model
model.to(device) #Moved model to device

train_data = [[x_train[i], y_train[i]] for i in range(len(y_train))]
test_data = (x_test, y_test)
validation_data = (x_val, y_val)

tokenized_train = tokenizer(x_train, return_tensors="pt", truncation = True, padding = True, max_length = 512).to(device)
tokenized_test = tokenizer(x_test, return_tensors="pt", truncation = True, padding = True, max_length = 512).to(device)
tokenized_val = tokenizer(x_val, return_tensors="pt", truncation = True, padding = True, max_length = 512).to(device)

dataloaded_train = DataLoader(train_data, batch_size = 32, shuffle = True)
data_loader_test = DataLoader(test_data, batch_size = 32, shuffle = False)
data_loader_val = DataLoader(validation_data, batch_size = 32, shuffle = False)


lora_config = LoraConfig(r = 8, target_modules = ["query_proj", "key_proj", "value_proj"]) #Double check the names of the  target modules, HOw do I do that?

model = get_peft_model(model, lora_config)

for batch in dataloaded_train:
    optimizer = AdamW
    #Actual training
    model.train()
    out = model(input_ids = tokenized_train["input_ids"], labels = dataloaded_train["labels"], attention_mask = tokenized_train["attention_mask"])
    loss = out.loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()


In [ ]:
for param in bert.parameters():
    param.requires_grad = False  #Freezes BERT

class BERT_architecture(nn.Module):

In [ ]:
model = make_pipeline(
    CountVectorizer(ngram_range=(1,2)),
    MultinomialNB()
)

vectorizer = CountVectorizer(ngram_range=(1,2))
x_train_vectorized = vectorizer.fit_transform(x_train)
model.fit(x_train, y_train)

#Good idea to implement the perceptron rule myself

